# ⛈ Cloudburst Detection System — Exploratory Data Analysis
This notebook covers:
1. Dataset exploration and statistics
2. Feature distribution analysis
3. Correlation analysis
4. Model training and evaluation
5. Feature importance visualization


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'monospace'
print('Libraries loaded ✅')

## 1. Generate + Load Dataset

In [ ]:
from models.train import generate_synthetic_data

df = generate_synthetic_data(n_samples=5000)
print(f'Shape: {df.shape}')
print(f'Cloudburst rate: {df.cloudburst_event.mean():.1%}')
df.head()

## 2. Feature Distributions by Class

In [ ]:
features_to_plot = ['rainfall_mm_1h','humidity_pct','pressure_hpa','wind_speed_kmh','radar_reflectivity_dbz','cape_proxy']
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
colors = {0: '#185FA5', 1: '#E24B4A'}
labels_map = {0: 'Normal', 1: 'Cloudburst'}

for ax, feat in zip(axes.flat, features_to_plot):
    for label in [0, 1]:
        data = df[df['cloudburst_event'] == label][feat]
        ax.hist(data, bins=40, alpha=0.6, color=colors[label], label=labels_map[label], density=True)
    ax.set_title(feat, fontsize=10)
    ax.legend(fontsize=8)
    ax.set_xlabel('')

plt.suptitle('Feature Distributions: Normal vs Cloudburst Events', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../models/results/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Correlation Heatmap

In [ ]:
num_cols = ['temperature_c','humidity_pct','pressure_hpa','wind_speed_kmh',
            'rainfall_mm_1h','rainfall_mm_3h','radar_reflectivity_dbz',
            'cape_proxy','instability_index','risk_score','cloudburst_event']
corr = df[num_cols].corr()

plt.figure(figsize=(12,10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title('Pearson Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.savefig('../models/results/correlation_heatmap.png', dpi=150)
plt.show()

## 4. Train Random Forest & Evaluate

In [ ]:
from models.train import generate_synthetic_data
from sklearn.preprocessing import RobustScaler

df = generate_synthetic_data(8000)
feature_cols = [c for c in df.columns if c != 'cloudburst_event']

X = RobustScaler().fit_transform(df[feature_cols])
y = df['cloudburst_event'].values

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_tr, y_tr)

proba = rf.predict_proba(X_te)[:, 1]
preds = (proba >= 0.35).astype(int)

print('Classification Report (threshold=0.35):')
print(classification_report(y_te, preds, target_names=['Normal','Cloudburst']))
print(f'AUC-ROC: {roc_auc_score(y_te, proba):.4f}')

## 5. Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=True).tail(20)

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(importance_df['feature'], importance_df['importance'], color='#185FA5', alpha=0.85)
ax.set_xlabel('Gini Importance')
ax.set_title('Top 20 Feature Importances — Random Forest')
for bar, val in zip(bars, importance_df['importance']):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('../models/results/feature_importance.png', dpi=150)
plt.show()